# Datasets and DataLoaders
`Dataset` defines how to access data. `DataLoader` wraps it to serve shuffled mini-batches during training.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import numpy as np

## 1. TensorDataset — quickest way

In [ ]:
X = torch.randn(100, 5)   # 100 samples, 5 features
y = torch.randint(0, 2, (100,))  # binary labels

dataset = TensorDataset(X, y)
print(len(dataset))          # 100
print(dataset[0])            # (tensor of shape [5], label tensor)

## 2. Custom Dataset
Subclass `Dataset`, implement `__len__` and `__getitem__`.

In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


X = torch.randn(200, 8)
y = (X[:, 0] > 0).float()   # label = sign of first feature

ds = SimpleDataset(X, y)
print(len(ds))
print(ds[0][0].shape, ds[0][1])

## 3. DataLoader — batching and shuffling

In [ ]:
loader = DataLoader(ds, batch_size=32, shuffle=True)

print(f"Total batches: {len(loader)}")   # ceil(200 / 32) = 7

for batch_X, batch_y in loader:
    print(batch_X.shape, batch_y.shape)
    break  # just inspect first batch

In [ ]:
# drop_last=True — drop final incomplete batch (useful for fixed batch-norm)
loader_strict = DataLoader(ds, batch_size=32, shuffle=True, drop_last=True)
print(f"Batches with drop_last: {len(loader_strict)}")  # 6, not 7

## 4. Train / Validation Split

In [ ]:
train_size = int(0.8 * len(ds))
val_size = len(ds) - train_size

train_ds, val_ds = random_split(ds, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)  # no shuffle for val

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

## 5. Typical Training Loop Skeleton

In [ ]:
import torch.nn as nn

model = nn.Linear(8, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(3):
    model.train()
    total_loss = 0.0

    for batch_X, batch_y in train_loader:
        preds = model(batch_X).squeeze()
        loss = criterion(preds, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}  train_loss={total_loss/len(train_loader):.4f}")

## 6. Transforms — preprocessing in the Dataset

In [ ]:
class NormalizedDataset(Dataset):
    """Normalizes X to zero mean, unit std on construction."""
    def __init__(self, X, y):
        self.X = (X - X.mean(dim=0)) / (X.std(dim=0) + 1e-8)
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


norm_ds = NormalizedDataset(X, y)
print(norm_ds.X.mean(dim=0).abs().max().item())  # ≈ 0